In [ ]:
# Unzip the dataset
import zipfile
import os

zip_path = "/content/archive.zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted to:", extract_path)

# Show folder structure
for root, dirs, files in os.walk(extract_path):
    level = root.replace(extract_path, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files[:5]:  # show only first 5 files per folder
        print(f"{indent}  {f}")


In [ ]:
# Basic dataset exploration
import os
import matplotlib.pyplot as plt
from PIL import Image
import random

dataset_path = "/content/dataset"
classes = os.listdir(dataset_path)

print("Classes:", classes)

# Count images per class
class_counts = {}
for cls in classes:
    class_counts[cls] = len(os.listdir(os.path.join(dataset_path, cls)))

print("\nImage count per class:")
for cls, count in class_counts.items():
    print(f"{cls}: {count}")


In [ ]:
# Visualize sample images from each class
plt.figure(figsize=(12, 8))

for i, cls in enumerate(classes):
    img_name = random.choice(os.listdir(os.path.join(dataset_path, cls)))
    img_path = os.path.join(dataset_path, cls, img_name)

    img = Image.open(img_path)

    plt.subplot(2, 2, i + 1)
    plt.imshow(img)
    plt.title(cls)
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# Install required libraries
!pip install -q tensorflow scikit-learn seaborn


In [ ]:
# Imports
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
import numpy as np


In [ ]:
# Collect image paths and labels
image_paths = []
labels = []

for cls in classes:
    class_dir = os.path.join(dataset_path, cls)
    for img in os.listdir(class_dir):
        image_paths.append(os.path.join(class_dir, img))
        labels.append(cls)

image_paths = np.array(image_paths)
labels = np.array(labels)

print("Total images:", len(image_paths))


In [ ]:
# Train / Validation / Test split (70 / 15 / 15)
X_temp, X_test, y_temp, y_test = train_test_split(
    image_paths, labels, test_size=0.15, stratify=labels, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=42
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))


In [ ]:
# Image size and batch size
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Training data generator (with augmentation)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.1,
    horizontal_flip=True
)

# Validation & test generators (NO augmentation)
val_test_datagen = ImageDataGenerator(rescale=1./255)


In [ ]:
# Create DataFrames for generators
import pandas as pd

train_df = pd.DataFrame({
    "filename": X_train,
    "class": y_train
})

val_df = pd.DataFrame({
    "filename": X_val,
    "class": y_val
})

test_df = pd.DataFrame({
    "filename": X_test,
    "class": y_test
})

print("Train DF shape:", train_df.shape)
print("Val DF shape:", val_df.shape)
print("Test DF shape:", test_df.shape)



In [ ]:
def create_generator(datagen, df, shuffle=True):
    return datagen.flow_from_dataframe(
        dataframe=df,
        x_col="filename",
        y_col="class",
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        shuffle=shuffle
    )

train_gen = create_generator(train_datagen, train_df, shuffle=True)
val_gen   = create_generator(val_test_datagen, val_df, shuffle=False)
test_gen  = create_generator(val_test_datagen, test_df, shuffle=False)


In [ ]:
# Build Transfer Learning Model (ResNet50)
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam


In [ ]:
# Load pretrained ResNet50 without the top classifier
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze base model layers
for layer in base_model.layers:
    layer.trainable = False

# Custom classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(4, activation="softmax")(x)

model_tl = Model(inputs=base_model.input, outputs=output)

model_tl.summary()


In [ ]:
# Compile model
model_tl.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=3,
    min_lr=1e-6
)


In [ ]:
history_tl = model_tl.fit(
    train_gen,
    validation_data=val_gen,
    epochs=25,
    callbacks=[early_stop, lr_scheduler]
)


In [ ]:
# Plot training vs validation curves
import matplotlib.pyplot as plt

def plot_history(history, title):
    plt.figure(figsize=(12,5))

    plt.subplot(1,2,1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Val Accuracy')
    plt.title(f"{title} - Accuracy")
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title(f"{title} - Loss")
    plt.legend()

    plt.show()

plot_history(history_tl, "ResNet50 Transfer Learning")


In [ ]:
# Check files in /content
import os

print(os.listdir("/content"))
